## 4. Production Pipeline: Full Preprocessing with ColumnTransformer

In real-world projects, we build a single unified preprocessing pipeline:
1. **Nominal Features** $\to$ `OneHotEncoder(handle_unknown='ignore')`
2. **Ordinal Features** $\to$ `OrdinalEncoder(categories=[...])`
3. **High-Cardinality Features** $\to$ `TargetEncoder(cv=3, smooth='auto')`
4. **Numerical Features** $\to$ `StandardScaler()` (or `RobustScaler()` if extreme outliers exist)
5. **No Data Leakage** $\to$ `fit_transform` on `X_train`, `transform` on `X_test`.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# 1. Realistic enterprise dataset with mixed feature types
data = {
    'Age': [25, 34, 45, 22, 28, 52, 40, 29, 36, 48, 23, 41],
    'Annual_Income': [35000, 78000, 120000, 28000, 65000, 140000, 95000, 52000, 88000, 110000, 31000, 99000],
    'Credit_Score': [610, 720, 790, 580, 690, 820, 750, 640, 710, 800, 600, 740],
    'Device_Type': ['Android', 'iOS', 'Android', 'Windows', 'iOS', 'iOS', 'Android', 'Windows', 'iOS', 'Android', 'Android', 'iOS'],
    'Education_Level': ['High School', 'Bachelors', 'PhD', 'High School', 'Masters', 'PhD', 'Bachelors', 'Masters', 'Bachelors', 'PhD', 'High School', 'Masters'],
    'City_Pincode': ['500001', '560001', '400001', '500001', '560001', '110001', '500001', '400001', '560001', '600001', '500001', '110001'],
    'Loan_Approved': [0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1]  # Target (y)
}

df = pd.DataFrame(data)
print("=== RAW INPUT DATASET ===")
display(df.head())

=== RAW INPUT DATASET ===


,Age,Annual_Income,Credit_Score,Device_Type,Education_Level,City_Pincode,Loan_Approved
0,25,35000,610,Android,High School,500001,0
1,34,78000,720,iOS,Bachelors,560001,1
2,45,120000,790,Android,PhD,400001,1
3,22,28000,580,Windows,High School,500001,0
4,28,65000,690,iOS,Masters,560001,1


---
## Step 1: Train-Test Split (Mandatory First Step)
Always split your data before fitting any encoders or scalers to prevent data leakage.

In [3]:
X = df.drop(columns=['Loan_Approved']).copy()
y = df['Loan_Approved'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")

Training rows: 9 | Test rows: 3


---
## Step 2: Build the Unified `ColumnTransformer`

Group columns by their preprocessing strategy:
* `num_cols`: Scaled with `StandardScaler()`
* `nominal_cols`: Encoded with `OneHotEncoder()`
* `ordinal_cols`: Encoded with `OrdinalEncoder()`
* `high_card_cols`: Encoded with `TargetEncoder()`

In [4]:
num_columns = ['Age', 'Annual_Income', 'Credit_Score']

preprocessor = ColumnTransformer(
    transformers=[
        ('num_scaler', StandardScaler(), num_columns),
        ('nominal_ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['Device_Type']),
        ('ordinal_enc', OrdinalEncoder(categories=[education_order]), ),
        ('target_enc', TargetEncoder(cv=3, smooth='auto', random_state=42), high_card_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')
X_train_transformed = ct.fit_transform(X_train, y_train)

X_test_transformed = ct.fit_transform(X_test)

print("=== TRANSFORMED TRAINING FEATURES (ENCODED & SCALED) ===")
display(X_train_transformed.head())

categories=[['High School','Bachelors', 'PhD', 'Masters']]

ValueError: not enough values to unpack (expected 3, got 2)